# 05 · Measurements and Density Matrices

**Covers:** density matrices, pure vs mixed states, purity, the partial trace, Bell state vs classical correlations, loss and detector efficiency, and single-qubit state tomography.
**Photonics thread:** real sources are noisy, real chips lose light, and real detectors miss photons. A density matrix describes all of that, and tomography is how a lab reconstructs it from clicks.

In [1]:
import numpy as np
import qutip as qt

results = []   # every theory-vs-simulation check is recorded here

def check(label, sim, theory, tol=1e-3):
    """Compare a simulated number with its analytic value and record the result."""
    sim, theory = float(np.real(sim)), float(np.real(theory))
    ok = bool(np.isclose(sim, theory, atol=tol))
    results.append(ok)
    print(f"{'PASS' if ok else 'FAIL'}  {label}: simulation = {sim:.4f}, theory = {theory:.4f}")

In [2]:
H, V = qt.basis(2, 0), qt.basis(2, 1)
plus = (H + V).unit()

def purity(rho):
    return (rho * rho).tr().real

## 1. Density matrix: a state that remembers what you don't know

**Plain words:** a state vector is a full recipe; a density matrix $\rho$ can also say "I'm not sure which recipe was used". For a pure state $\rho = |\psi\rangle\langle\psi|$.

- **Diagonal:** the probabilities of H and V.
- **Off-diagonal:** the **coherence**, the part that makes interference possible.

A 45° photon and a source that sends H or V at random have the *same* diagonal, but only the 45° photon has off-diagonals. A 45° polarizer tells them apart: 100 % vs 50 % transmission. This settles "is a superposition just hidden H-or-V?": no.

**Purity** $\mathrm{Tr}(\rho^2)$ is 1 for a pure state and ½ for a maximally mixed qubit.

**Gap:** different mixtures can give the same $\rho$ (half H/V and half 45°/135° both give $I/2$). $\rho$ describes what can be predicted, not how the state was made.

**One sentence:** the diagonal gives the H/V odds, the off-diagonal says whether interference is possible.

In [3]:
rho_pure = qt.ket2dm(plus)
rho_mix = 0.5 * qt.ket2dm(H) + 0.5 * qt.ket2dm(V)
print("pure 45 deg:\n", rho_pure.full().real.round(3))
print("mixed H/V:\n", rho_mix.full().real.round(3))
proj45 = qt.ket2dm(plus)
check("45 deg polarizer, pure state", qt.expect(proj45, rho_pure), 1.0)
check("45 deg polarizer, mixed state", qt.expect(proj45, rho_mix), 0.5)
check("purity, pure", purity(rho_pure), 1.0)
check("purity, mixed", purity(rho_mix), 0.5)

pure 45 deg:
 [[0.5 0.5]
 [0.5 0.5]]
mixed H/V:
 [[0.5 0. ]
 [0.  0.5]]
PASS  45 deg polarizer, pure state: simulation = 1.0000, theory = 1.0000
PASS  45 deg polarizer, mixed state: simulation = 0.5000, theory = 0.5000
PASS  purity, pure: simulation = 1.0000, theory = 1.0000
PASS  purity, mixed: simulation = 0.5000, theory = 0.5000


## 2. Partial trace: one photon of an entangled pair

**Plain words:** keep photon 1, ignore photon 2, and sum over everything photon 2 could have been. For the Bell state the result is $\rho_1 = I/2$: the pair is perfectly known, yet each photon alone is maximally mixed. All the information lives in the correlations.

**Photonics:** photon loss is a partial trace. A lost photon becomes entangled with the environment, and what you keep is mixed. One arm of an SPDC pair on its own is mixed for the same reason, which is why heralding matters.

**One sentence:** tracing out one photon of an entangled pair leaves the other completely random.

In [4]:
bell = (qt.tensor(H, H) + qt.tensor(V, V)).unit()
rho1 = qt.ket2dm(bell).ptrace(0)
print("photon 1 alone:\n", rho1.full().real.round(3))
check("purity of the Bell pair", purity(qt.ket2dm(bell)), 1.0)
check("purity of one photon of the pair", purity(rho1), 0.5)

photon 1 alone:
 [[0.5 0. ]
 [0.  0.5]]
PASS  purity of the Bell pair: simulation = 1.0000, theory = 1.0000
PASS  purity of one photon of the pair: simulation = 0.5000, theory = 0.5000


## 3. Bell pair vs pre-agreed coins

A classical source that always sends HH or VV matches every H/V measurement, just like the Bell state. Measure **both photons at 45°** instead:

- Bell state: $P(++) = 0.5$
- Classical coins: $P(++) = 0.25$

**One sentence:** entanglement is correlation that survives in every measurement basis, not just one.

In [5]:
rho_cl = 0.5 * qt.ket2dm(qt.tensor(H, H)) + 0.5 * qt.ket2dm(qt.tensor(V, V))
pp = qt.ket2dm(qt.tensor(plus, plus))
check("P(++) for the Bell state", qt.expect(pp, qt.ket2dm(bell)), 0.5)
check("P(++) for classical coins", qt.expect(pp, rho_cl), 0.25)

PASS  P(++) for the Bell state: simulation = 0.5000, theory = 0.5000
PASS  P(++) for classical coins: simulation = 0.2500, theory = 0.2500


## 4. Loss is a beam splitter plus a partial trace

**Plain words:** lost light leaks into a mode you don't watch. Model: a beam splitter (transmission η) mixes the signal with an empty hidden mode, then the hidden mode is traced out. A detector of efficiency η is "lose some light, then detect perfectly".

| Light | After loss η |
|---|---|
| Laser | still a laser, α → √η α |
| Single photon | mixed: η\|1⟩⟨1\| + (1−η)\|0⟩⟨0\|, Wigner negativity gone once η ≤ 0.5 |
| Squeezed vacuum | noise η e^{−2r} + (1−η) in vacuum units |

$g^{(2)}(0)$ is unchanged by loss (numerator and denominator both shrink by η²), so it is a loss-tolerant figure of merit; squeezing and Wigner negativity are loss-fragile.

**Gap:** this ideal model has no dark counts or background light, which do raise $g^{(2)}$ in real data.

**One sentence:** loss mixes vacuum into your state, so lasers survive it and nonclassical features fade.

In [6]:
N = 25
a, n = qt.destroy(N), qt.num(N)
A = qt.tensor(qt.destroy(N), qt.qeye(N))
B = qt.tensor(qt.qeye(N), qt.destroy(N))

def lossy(state, eta):
    theta = np.arccos(np.sqrt(eta))                       # eta = cos^2(theta)
    U = (theta * (A.dag() * B - A * B.dag())).expm()
    rho = state if state.isoper else qt.ket2dm(state)
    both = qt.tensor(rho, qt.ket2dm(qt.basis(N, 0)))     # hidden mode starts empty
    return (U * both * U.dag()).ptrace(0)

def g2(rho):
    return qt.expect(a.dag() * a.dag() * a * a, rho) / qt.expect(n, rho) ** 2

xv = np.linspace(-5, 5, 201)
one = qt.basis(N, 1)
for eta in [1.0, 0.8, 0.5, 0.3]:
    rho = lossy(one, eta)
    check(f"single photon, eta={eta}: pi W(0,0) = 1 - 2 eta", qt.wigner(rho, xv, xv)[100, 100] * np.pi, 1 - 2 * eta)

rc = lossy(qt.coherent(N, 2), 0.5)
check("laser after 50% loss: <n>", qt.expect(n, rc), 2.0)
check("laser after 50% loss: Var(n) = <n>", qt.variance(n, rc), 2.0)
check("click probability, mean 1 photon, eta 0.5", 1 - lossy(qt.coherent(N, 1), 0.5).diag()[0], 1 - np.exp(-0.5))

sq = qt.squeeze(N, 0.5) * qt.basis(N, 0)
rs = lossy(sq, 0.5)
check("g2 of squeezed vacuum survives loss", g2(rs), g2(qt.ket2dm(sq)))
x = (a + a.dag()) / np.sqrt(2)
check("squeezed noise after loss (vacuum units)", qt.variance(x, rs) / 0.5, 0.5 * np.exp(-1) + 0.5)

PASS  single photon, eta=1.0: pi W(0,0) = 1 - 2 eta: simulation = -1.0000, theory = -1.0000


PASS  single photon, eta=0.8: pi W(0,0) = 1 - 2 eta: simulation = -0.6000, theory = -0.6000


PASS  single photon, eta=0.5: pi W(0,0) = 1 - 2 eta: simulation = 0.0000, theory = 0.0000


PASS  single photon, eta=0.3: pi W(0,0) = 1 - 2 eta: simulation = 0.4000, theory = 0.4000


PASS  laser after 50% loss: <n>: simulation = 2.0000, theory = 2.0000
PASS  laser after 50% loss: Var(n) = <n>: simulation = 2.0000, theory = 2.0000


PASS  click probability, mean 1 photon, eta 0.5: simulation = 0.3935, theory = 0.3935


PASS  g2 of squeezed vacuum survives loss: simulation = 6.6827, theory = 6.6827
PASS  squeezed noise after loss (vacuum units): simulation = 0.6839, theory = 0.6839


## 5. Tomography: rebuilding ρ from clicks

**Plain words:** a state can't be photographed, but it can be rebuilt from shadows cast in three directions. For one qubit, measure X (45/135°), Y (circular) and Z (H/V):

$$\rho = \tfrac12\left(I + \langle X\rangle X + \langle Y\rangle Y + \langle Z\rangle Z\right)$$

Each average comes from a finite number of clicks, with error about $1/\sqrt{\text{shots}}$.

**Gap:** with few shots the noisy averages can describe an impossible state (a negative eigenvalue, "fidelity" above 1). Labs fix this with maximum-likelihood fitting, not done here.

**One sentence:** tomography rebuilds ρ from three kinds of questions, and its accuracy is limited by the number of clicks.

In [7]:
psi = qt.Qobj([[0.6], [0.8]])
X, Y, Z, I2 = qt.sigmax(), qt.sigmay(), qt.sigmaz(), qt.qeye(2)

def rebuild(mx, my, mz):
    return 0.5 * (I2 + mx * X + my * Y + mz * Z)

exact = rebuild(*[qt.expect(s, psi) for s in (X, Y, Z)])
print(exact.full().real.round(3))
check("exact tomography returns |psi><psi|", qt.fidelity(exact, qt.ket2dm(psi)), 1.0)

rng = np.random.default_rng(1)                            # fixed seed: reproducible
def measure(op, shots):
    prob = np.clip((1 + qt.expect(op, psi)) / 2, 0, 1)
    return 2 * rng.binomial(shots, prob) / shots - 1

for shots in [100, 1000, 100000]:
    est = rebuild(measure(X, shots), measure(Y, shots), measure(Z, shots))
    print(f"{shots:>6} shots: <psi|rho_est|psi> = {qt.expect(qt.ket2dm(psi), est):.4f}, "
          f"smallest eigenvalue = {min(est.eigenenergies()):+.4f}")

[[0.36 0.48]
 [0.48 0.64]]
PASS  exact tomography returns |psi><psi|: simulation = 1.0000, theory = 1.0000
   100 shots: <psi|rho_est|psi> = 1.0196, smallest eigenvalue = -0.0375
  1000 shots: <psi|rho_est|psi> = 1.0077, smallest eigenvalue = -0.0078
100000 shots: <psi|rho_est|psi> = 0.9995, smallest eigenvalue = +0.0005


With 100 shots the overlap can exceed 1 and the smallest eigenvalue can be negative: an unphysical estimate, as described in the gap above.

## Validation summary

Every `check` above compared a QuTiP result with its analytic formula.

In [8]:
print(f"{sum(results)} of {len(results)} checks passed")
assert all(results), "a simulation disagrees with theory"

18 of 18 checks passed
